# Advanced Python Reference Management — 26 Solved Problems

**Main subject:** strong references, weak references, `weakref.ref`, `WeakKeyDictionary`, weak-reference containers, descriptor storage, and lifetime-safe object design. **Level:** intermediate → advanced. **Format:** challenge → worked reasoning → runnable solution → assertions. All code uses the standard library.

**Learning goals:** trace ownership without dereferencing freed memory; distinguish weak vs strong access; design weak-key descriptor storage; handle equality, hashing, unhashable objects, slots, caches, callbacks, finalizers, weak bound methods, and thread-safe lifetime checks.

**Run order:** Kernel → Restart & Run All. Work on each challenge before revealing the solution. All problem solutions are self-contained except Problem 21, which explicitly builds on the `IdentityWeakMap` defined in Problem 20. In a notebook, only source code is stored, not fabricated execution results.

## Source alignment and corrections

The supplied lesson introduces **strong vs weak references**, demonstrates `weakref.ref()` and its `None` result after collection, explains why a local assignment of `ref()` makes a **new strong reference**, introduces `weakref.getweakrefcount()`, shows the weak-key dictionary and its application to **data descriptors**, and flags that weak keys must be **weak-referenceable and hashable**.

**Safety / correctness improvements for these exercises:**

- Never read a memory address via `ctypes.c_long.from_address(id(obj))` after the object dies: this is undefined/unsafe. We use weak references to detect lifetime instead.
- On CPython, ordinary acyclic objects are usually reclaimed when their strong reference count hits zero; *cyclic garbage collection* also handles unreachable cycles. Other Python interpreters may defer reclamation. Where a test needs collection, it explicitly calls `gc.collect()` and checks state rather than exact reference counts.
- `sys.getrefcount()` is CPython-specific, includes a temporary argument reference, and counts may vary in notebooks or with interpreter optimizations. Treat counts as diagnostics, not portable invariants.
- Notebook display history (e.g., the last expression, `Out`, `_`) can retain objects. Demo functions return only primitive results and keep referents out of displayed output.
- A `WeakKeyDictionary` holds weak **keys** but strong **values**. A value that strongly references its own key can inadvertently keep that key alive. Also, equality-defined distinct keys can collapse into one entry; weak keys do not automatically give identity semantics.

**Environment:** Python 3.10+ recommended. Problem 26 includes an optional Python 3.11+ dataclass branch. There are no network calls, third-party dependencies, unsafe address reads, or assumptions that `gc.collect()` returns a fixed integer.

In [1]:
import gc
import platform
import sys
import threading
import weakref
from collections import OrderedDict
from dataclasses import dataclass
from typing import Any

print('Interpreter:', platform.python_implementation(), sys.version.split()[0])
print('weakref and gc are ready')

Interpreter: CPython 3.13.7
weakref and gc are ready


## Pre-flight conceptual example — a safe lifetime probe

A weak reference is **not** a second owner. Call it once and keep the returned referent in a local variable for the duration of an operation. `gc.collect()` is a test aid, not routine production code. The observable result below is independent of exact reference counts.

In [2]:
def safe_lifetime_probe():
    class Item:
        pass

    obj = Item()
    witness = weakref.ref(obj)
    alias = obj                     # another strong owner
    del obj                         # alias still owns the object
    assert witness() is alias
    del alias                       # drop last strong owner
    gc.collect()
    return witness() is None

assert safe_lifetime_probe()
print('Weak reference expires after strong ownership disappears.')

Weak reference expires after strong ownership disappears.


## Problem 01 — Alias topology and object lifetime

**Challenge.** Construct three strong variable bindings and one weak reference to the *same* object. Delete the aliases one at a time; record whether the weak reference still resolves. Do not return the referent or display it as a final expression.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 01 — reasoning

Names hold strong references; assignment does not clone the object. The weak reference is observational. Keep any temporary dereference confined to the boolean identity check.

In [3]:
def solve_01():
    class Node:
        pass

    first = Node()
    second = first
    third = second
    observer = weakref.ref(first)
    states = [observer() is first]

    del first
    states.append(observer() is second)
    del second
    states.append(observer() is third)
    del third
    gc.collect()
    states.append(observer() is None)
    return states

In [4]:
assert solve_01() == [True, True, True, True]
print("Problem 01: PASS")

Problem 01: PASS


## Problem 02 — Count strong references without brittle assertions

**Challenge.** Use `sys.getrefcount` on CPython to compare reference counts before and after creating one additional strong alias; use `weakref.getweakrefcount` to observe weak refs. Why should absolute counts never be hard-coded across interpreters or interactive execution contexts?

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 02 — reasoning

A newly created alias increases strong ownership; a weak reference does not. This experiment checks only a **relative** change when running on CPython. Exact counts and implementation support vary; `getrefcount` itself temporarily holds its argument.

In [5]:
def solve_02():
    class Record:
        pass

    owner = Record()
    weak = weakref.ref(owner)
    weak_before = weakref.getweakrefcount(owner)
    if platform.python_implementation() != 'CPython':
        return {'checked': False, 'weak_nonzero': weak_before >= 1}

    initial = sys.getrefcount(owner)
    alias = owner
    with_alias = sys.getrefcount(owner)
    del alias
    after = sys.getrefcount(owner)
    return {
        'checked': True,
        'alias_delta': with_alias - initial,
        'restored': after == initial,
        'weak_nonzero': weak_before >= 1,
        'weak_same_target': weak() is owner,
    }

In [6]:
result = solve_02()
assert result['weak_nonzero']
if result['checked']:
    assert result['alias_delta'] == 1 and result['restored']
    assert result['weak_same_target']
print("Problem 02: PASS")

Problem 02: PASS


## Problem 03 — Dereference exactly once

**Challenge.** Write `describe(ref)` that safely returns a record name or `None`. Explain why `if ref() is not None: return ref().name` is unsafe when object lifetime could change between two calls.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 03 — reasoning

Copy the result of `ref()` into a local variable, then test that variable. The local is a temporary **strong reference** while the operation completes; the naive two-call pattern has a time-of-check/time-of-use race.

In [7]:
def describe(ref: weakref.ReferenceType) -> str | None:
    item = ref()
    if item is None:
        return None
    return item.name


def solve_03():
    class Entry:
        def __init__(self, name):
            self.name = name

    owner = Entry('alive')
    witness = weakref.ref(owner)
    before = describe(witness)
    del owner
    gc.collect()
    after = describe(witness)
    return before, after

In [8]:
assert solve_03() == ('alive', None)
print("Problem 03: PASS")

Problem 03: PASS


## Problem 04 — Weak-reference eligibility and slots

**Challenge.** Try weakly referencing a plain `list`, a subclass of `list`, a class with restrictive `__slots__`, and a slotted class that adds `__weakref__`. Report only booleans; avoid uncaught `TypeError`.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 04 — reasoning

Most plain `list` instances cannot be weakly referenced. A typical user-defined subclass is weak-referenceable. Classes with `__slots__` normally need the explicit `__weakref__` slot unless it is supplied by a base class.

In [9]:
def solve_04():
    class WeakList(list):
        pass

    class SlottedNoWeak:
        __slots__ = ('x',)

    class SlottedWithWeak:
        __slots__ = ('x', '__weakref__')

    def supported(value):
        try:
            return weakref.ref(value)() is value
        except TypeError:
            return False

    return {
        'plain_list': supported([]),
        'list_subclass': supported(WeakList([1, 2])),
        'slots_without_weakref': supported(SlottedNoWeak()),
        'slots_with_weakref': supported(SlottedWithWeak()),
    }

In [10]:
assert solve_04() == {
    'plain_list': False,
    'list_subclass': True,
    'slots_without_weakref': False,
    'slots_with_weakref': True,
}
print("Problem 04: PASS")

Problem 04: PASS


## Problem 05 — Weak-reference callbacks and cleanup

**Challenge.** Create a callback that logs an identifier when an object is reclaimed. Ensure the callback does not close over the referent, and distinguish callback notification from any guarantee about its exact execution time.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 05 — reasoning

A callback receives the **weak-reference object**, not the dead referent. Capture an immutable label and log it; do not capture `obj` in the callback. Call `gc.collect()` to make this deterministic for the test; exact collection timing is implementation-dependent.

In [11]:
def solve_05():
    events = []

    class Document:
        pass

    owner = Document()
    label = 'document-17'

    def on_collect(dead_reference):
        events.append((label, dead_reference() is None))

    watcher = weakref.ref(owner, on_collect)
    assert watcher() is owner
    del owner
    gc.collect()
    return events, watcher() is None

In [12]:
events, dead = solve_05()
assert dead is True
assert events == [('document-17', True)]
print("Problem 05: PASS")

Problem 05: PASS


## Problem 06 — Dead weak references and defensive APIs

**Challenge.** Implement an `alive(ref)` predicate and an `access(ref, default)` helper that never accidentally accesses a dead referent. Check that weak-reference objects themselves remain usable after death.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 06 — reasoning

Calling a dead weak reference returns `None`, but the weak-reference object may survive. Guard dereferencing by assigning the result once. A convenience API can return a caller-provided fallback instead of throwing an attribute error.

In [13]:
def alive(ref):
    obj = ref()
    return obj is not None


def access(ref, default='<missing>'):
    obj = ref()
    return default if obj is None else obj.message


def solve_06():
    class Notice:
        message = 'received'

    owner = Notice()
    ref = weakref.ref(owner)
    live = (alive(ref), access(ref))
    del owner
    gc.collect()
    gone = (alive(ref), access(ref))
    return live, gone, isinstance(ref, weakref.ReferenceType)

In [14]:
live, gone, ref_still_exists = solve_06()
assert live == (True, 'received')
assert gone == (False, '<missing>')
assert ref_still_exists
print("Problem 06: PASS")

Problem 06: PASS


## Problem 07 — Weak-key automatic eviction

**Challenge.** Put two identity-hashable objects into a `WeakKeyDictionary`. Delete one owner, force collection, then verify only its entry disappears. Do not collect the keys into a list and accidentally keep them alive.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 07 — reasoning

The mapping holds weak references to keys and strong references to values. The survivor has an external owner. Inspect only `len(mapping)` and the surviving entry, not a list of keys that would create additional strong ownership.

In [15]:
def solve_07():
    class Session:
        pass

    table = weakref.WeakKeyDictionary()
    left, right = Session(), Session()
    left_ref = weakref.ref(left)
    right_ref = weakref.ref(right)
    table[left] = 'left-state'
    table[right] = 'right-state'
    before = len(table)
    del left
    gc.collect()
    middle = (len(table), left_ref() is None, table[right])
    del right
    gc.collect()
    end = (len(table), right_ref() is None)
    return before, middle, end

In [16]:
before, middle, end = solve_07()
assert before == 2
assert middle == (1, True, 'right-state')
assert end == (0, True)
print("Problem 07: PASS")

Problem 07: PASS


## Problem 08 — A validating data descriptor backed by weak keys

**Challenge.** Implement `PositiveNumber` with `__set_name__`, `__get__`, `__set__`, and `__delete__`. Use `WeakKeyDictionary` for per-instance storage, validate strictly positive real `int`/`float` values but reject `bool`, and raise `AttributeError` on missing values. Verify instances can be reclaimed.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 08 — reasoning

A data descriptor centralizes validation, avoids storing a strong instance-key map on the class, and handles class-level access with `instance is None`. The owner must be hashable and weak-referenceable. Do not let stored values retain their owners.

In [17]:
class PositiveNumber:
    def __init__(self):
        self._values = weakref.WeakKeyDictionary()
        self._name = None

    def __set_name__(self, owner, name):
        self._name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._values[instance]
        except KeyError:
            raise AttributeError(f'{self._name} has not been set') from None

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f'{self._name} requires a real number')
        if not 0 < value < float('inf'):
            raise ValueError(f'{self._name} must be finite and positive')
        self._values[instance] = value

    def __delete__(self, instance):
        try:
            del self._values[instance]
        except KeyError:
            raise AttributeError(f'{self._name} has not been set') from None


class Invoice:
    amount = PositiveNumber()


def solve_08():
    inv = Invoice()
    assert Invoice.amount is Invoice.__dict__['amount']
    inv.amount = 12.5
    result = inv.amount
    watcher = weakref.ref(inv)
    invalid = []
    for value in (0, -1, float('inf'), True, '12'):
        try:
            inv.amount = value
        except (ValueError, TypeError) as exc:
            invalid.append(type(exc).__name__)
    del inv.amount
    try:
        _ = inv.amount
    except AttributeError:
        missing_works = True
    else:
        missing_works = False
    inv.amount = 3
    del inv
    gc.collect()
    return result, invalid, missing_works, len(Invoice.amount._values), watcher() is None

In [18]:
result, invalid, missing, size, dead = solve_08()
assert result == 12.5
assert invalid == ['ValueError', 'ValueError', 'ValueError', 'TypeError', 'TypeError']
assert missing and size == 0 and dead
print("Problem 08: PASS")

Problem 08: PASS


## Problem 09 — Equal objects collide as weak keys

**Challenge.** Create two different but equal, hashable `User` objects. Insert both into `WeakKeyDictionary` with different values. Explain why the mapping may have one entry and why removing the *first* object can remove the entry despite the second remaining alive.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 09 — reasoning

Weak-key dictionaries follow the referents’ hash/equality rules; they are **not identity maps**. Inserting an equal key updates the value while the original key object can remain the stored weak key. This is a common and surprising descriptor storage bug.

In [19]:
def solve_09():
    class User:
        def __init__(self, name):
            self.name = name

        def __eq__(self, other):
            return isinstance(other, User) and self.name == other.name

        def __hash__(self):
            return hash(self.name)

    first = User('same')
    second = User('same')
    table = weakref.WeakKeyDictionary()
    table[first] = 'initial'
    table[second] = 'updated'
    before = (first is not second, len(table), table[second])
    second_ref = weakref.ref(second)
    del first
    gc.collect()
    after = (len(table), second_ref() is second)
    return before, after

In [20]:
before, after = solve_09()
assert before == (True, 1, 'updated')
assert after == (0, True)
print("Problem 09: PASS")

Problem 09: PASS


## Problem 10 — Mutable hashes break weak-key lookup

**Challenge.** Build a weak-key class whose `__hash__` depends on a mutable field. Insert it, mutate the field, and show that lookup with the same object can fail. Then propose an identity-stable key design.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 10 — reasoning

Hash-based containers require a key’s hash to remain stable while stored. Using a mutable field violates that requirement. Avoid a strict assertion about the exact lookup result across implementations: the **broken invariant** is the point. Prefer default identity-based `object.__hash__`/`object.__eq__`, or stable immutable value equality.

In [21]:
def solve_10():
    class MutableKey:
        def __init__(self):
            self.code = 101

        def __hash__(self):
            return self.code

        def __eq__(self, other):
            return isinstance(other, MutableKey) and self.code == other.code

    key = MutableKey()
    table = weakref.WeakKeyDictionary()
    table[key] = 'payload'
    original_access = table[key]
    key.code = 202
    retrieved = table.get(key, '<missing>')
    witness = weakref.ref(key)
    del key
    gc.collect()
    return original_access, retrieved, len(table), witness() is None

In [22]:
original, retrieved, remaining, expired = solve_10()
assert original == 'payload'
assert retrieved in ('payload', '<missing>')   # lookup after hash mutation is unreliable
assert remaining == 0 and expired
print("Problem 10: PASS")

Problem 10: PASS


## Problem 11 — Dataclass hashability and weak keys

**Challenge.** Compare default mutable `@dataclass` equality with identity-based `@dataclass(eq=False)`. Make both weak-referenceable, and determine which is suitable as a `WeakKeyDictionary` key when each *instance* must have independent state.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 11 — reasoning

A default mutable dataclass generates equality but sets `__hash__ = None` unless explicitly customized. `eq=False` retains identity equality/hash for a normal user-defined class, making distinct, otherwise equal-looking instances independent keys.

In [23]:
def solve_11():
    @dataclass
    class EqualityPerson:
        name: str

    @dataclass(eq=False)
    class IdentityPerson:
        name: str

    equal_style = EqualityPerson('Ada')
    identity_a = IdentityPerson('Ada')
    identity_b = IdentityPerson('Ada')
    try:
        hash(equal_style)
    except TypeError:
        equality_unhashable = True
    else:
        equality_unhashable = False
    mapping = weakref.WeakKeyDictionary()
    mapping[identity_a] = 1
    mapping[identity_b] = 2
    return equality_unhashable, identity_a != identity_b, len(mapping), (
        mapping[identity_a], mapping[identity_b]
    )

In [24]:
assert solve_11() == (True, True, 2, (1, 2))
print("Problem 11: PASS")

Problem 11: PASS


## Problem 12 — Weak-value caches are not ownership mechanisms

**Challenge.** Build a `WeakValueDictionary` cache. Insert a value while a client owns it, verify cache retrieval; delete the client reference and observe cache eviction. Explain why `cache[key] = Resource()` may disappear immediately on CPython.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 12 — reasoning

A weak-value cache does **not** own resources. Only some other strong reference keeps entries usable. `WeakValueDictionary` keys are normal, strongly held dictionary keys; only the values are weak.

In [25]:
def solve_12():
    class Resource:
        def __init__(self, payload):
            self.payload = payload

    cache = weakref.WeakValueDictionary()
    resource = Resource('expensive result')
    pointer = weakref.ref(resource)
    cache['a'] = resource
    while_owned = (cache['a'] is resource, len(cache))
    del resource
    gc.collect()
    after = (cache.get('a') is None, len(cache), pointer() is None)
    return while_owned, after

In [26]:
assert solve_12() == ((True, 1), (True, 0, True))
print("Problem 12: PASS")

Problem 12: PASS


## Problem 13 — Use one strong local during cache lookup

**Challenge.** Implement `get_or_create(cache, key, factory)` for a `WeakValueDictionary` without the `if key in cache: return cache[key]` double-lookup pattern. Count factory calls and check that a live owner is reused.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 13 — reasoning

Checking presence and then looking up again can race with reclamation. `value = cache.get(key)` obtains a temporary strong owner for successful retrieval. Weak caches may evict between independent invocations and are not synchronization primitives for factories.

In [27]:
def get_or_create(cache, key, factory):
    value = cache.get(key)
    if value is None:
        value = factory()
        cache[key] = value
    return value


def solve_13():
    class Image:
        pass

    cache = weakref.WeakValueDictionary()
    calls = [0]

    def factory():
        calls[0] += 1
        return Image()

    first = get_or_create(cache, 'icon', factory)
    second = get_or_create(cache, 'icon', factory)
    reused = first is second
    del first, second
    gc.collect()
    was_evicted = cache.get('icon') is None
    third = get_or_create(cache, 'icon', factory)
    assert isinstance(third, Image)
    return reused, was_evicted, calls[0]

In [28]:
assert solve_13() == (True, True, 2)
print("Problem 13: PASS")

Problem 13: PASS


## Problem 14 — WeakSet-based observer registry

**Challenge.** Implement an event hub with a `WeakSet` of observer objects. Subscribe two observers; verify both receive the first event, delete one owner, and confirm only the remaining observer receives the next event.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 14 — reasoning

`WeakSet` does not keep subscribers alive. Observers must be weak-referenceable and hashable. Snapshot the set with `tuple(...)` during dispatch to avoid structural modification surprises; the snapshot temporarily owns observers while dispatch runs.

In [29]:
class ObserverHub:
    def __init__(self):
        self._observers = weakref.WeakSet()

    def subscribe(self, observer):
        self._observers.add(observer)

    def notify(self, event):
        for observer in tuple(self._observers):
            observer.on_event(event)

    def __len__(self):
        return len(self._observers)


def solve_14():
    received = []

    class Observer:
        def __init__(self, label):
            self.label = label

        def on_event(self, event):
            received.append((self.label, event))

    hub = ObserverHub()
    alpha, beta = Observer('a'), Observer('b')
    hub.subscribe(alpha)
    hub.subscribe(beta)
    hub.notify('first')
    first = sorted(received)
    alpha_ref = weakref.ref(alpha)
    del alpha
    gc.collect()
    hub.notify('second')
    return first, len(hub), alpha_ref() is None, sorted(received)

In [30]:
first, remaining, expired, all_events = solve_14()
assert first == [('a', 'first'), ('b', 'first')]
assert remaining == 1 and expired
assert all_events == [('a', 'first'), ('b', 'first'), ('b', 'second')]
print("Problem 14: PASS")

Problem 14: PASS


## Problem 15 — WeakMethod and ephemeral bound methods

**Challenge.** Why is `weakref.ref(listener.handle)` usually ineffective when `listener.handle` produces a temporary bound-method object? Use `weakref.WeakMethod` instead and verify it tracks the instance’s lifetime.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 15 — reasoning

Ordinary attribute access constructs a bound-method object that may disappear immediately when no one owns it. `WeakMethod` keeps weak ties to the instance/function and reconstructs a usable bound method while the instance is alive. The bound method returned from `WeakMethod()` is itself temporarily strong.

In [31]:
def solve_15():
    events = []

    class Listener:
        def handle(self, payload):
            events.append(payload)

    owner = Listener()
    ordinary = weakref.ref(owner.handle)
    ordinary_is_dead = ordinary() is None
    method_ref = weakref.WeakMethod(owner.handle)
    resolved = method_ref()
    assert resolved is not None
    resolved('hello')
    del resolved                      # do not accidentally keep the instance alive
    del owner
    gc.collect()
    return ordinary_is_dead, events, method_ref() is None

In [32]:
assert solve_15() == (True, ['hello'], True)
print("Problem 15: PASS")

Problem 15: PASS


## Problem 16 — Weak proxies and ReferenceError

**Challenge.** Create a `weakref.proxy(obj)` and demonstrate convenient attribute access while the object is alive, followed by a `ReferenceError` when it dies. Decide where explicit `ref()` checks are preferable.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 16 — reasoning

A proxy forwards attribute access while alive; it raises `ReferenceError` after collection rather than returning `None`. A regular weak reference is preferable where absence is routine and must be handled gracefully.

In [33]:
def solve_16():
    class Service:
        status = 'ready'

    owner = Service()
    proxy = weakref.proxy(owner)
    before = proxy.status
    del owner
    gc.collect()
    try:
        _ = proxy.status
    except ReferenceError:
        failed_as_expected = True
    else:
        failed_as_expected = False
    return before, failed_as_expected

In [34]:
assert solve_16() == ('ready', True)
print("Problem 16: PASS")

Problem 16: PASS


## Problem 17 — Safe finalizers: do not capture the owner

**Challenge.** Register a cleanup task with `weakref.finalize` using an external resource and an unbound function. Verify cleanup occurs after the owner dies, once only, and `finalizer.alive` becomes false.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 17 — reasoning

`weakref.finalize(owner, callback, ...)` must not have a callback/argument that strongly points back to `owner`. Give it a separate handle that is independent of the owner. The finalizer holds the callback and its arguments strongly by design.

In [35]:
def solve_17():
    events = []

    class Handle:
        def __init__(self, label):
            self.label = label

        def close(self):
            events.append(('closed', self.label))

    class Owner:
        pass

    handle = Handle('H-1')
    owner = Owner()
    fin = weakref.finalize(owner, Handle.close, handle)
    before = fin.alive
    owner_ref = weakref.ref(owner)
    del owner
    gc.collect()
    after = (fin.alive, owner_ref() is None, list(events))
    fin()                             # a completed finalizer does not run twice
    return before, after, events

In [36]:
before, after, events = solve_17()
assert before
assert after == (False, True, [('closed', 'H-1')])
assert events == [('closed', 'H-1')]
print("Problem 17: PASS")

Problem 17: PASS


## Problem 18 — Finalizer bound-method retention trap

**Challenge.** Intentionally register `weakref.finalize(owner, owner.cleanup)` and show that retaining the finalizer can keep the owner alive. Fix by calling `detach()` and dropping the returned tuple immediately; in production, use the external-handle pattern of Problem 17.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 18 — reasoning

A bound method strongly references `owner`, and the global finalizer registry retains its callback. This is a self-retention trap, not a reliable garbage-collection cleanup strategy. `detach()` removes the registration; avoid storing its returned tuple because it contains the owner and bound method.

In [37]:
def solve_18():
    events = []

    class Owner:
        def cleanup(self):
            events.append('cleanup')

    obj = Owner()
    watcher = weakref.ref(obj)
    finalizer = weakref.finalize(obj, obj.cleanup)  # intentionally flawed
    del obj
    gc.collect()
    retained = watcher() is not None
    finalizer.detach()                # do not bind the resulting tuple to a variable
    gc.collect()
    released = watcher() is None
    return retained, released, finalizer.alive, events

In [38]:
retained, released, still_registered, events = solve_18()
assert retained and released
assert still_registered is False and events == []
print("Problem 18: PASS")

Problem 18: PASS


## Problem 19 — Strong reference cycles versus weak edges

**Challenge.** Make two nodes point strongly to one another and observe that they become unreachable after their external owners are deleted. Compare that with one weak edge that does not create a strong cycle. Use `gc.collect()` in tests without asserting an exact collection count.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 19 — reasoning

A reference count of zero is not the only route to reclamation: tracing cyclic garbage collection can recover unreachable strong cycles. Turning one edge into a weak ref breaks the strong ownership cycle. Both designs should be unreachable once external owners vanish.

In [39]:
def solve_19():
    class Node:
        def __init__(self):
            self.peer = None

    first, second = Node(), Node()
    first.peer = second
    second.peer = first
    cyclic_refs = weakref.ref(first), weakref.ref(second)
    del first, second
    gc.collect()
    strong_cycle_reclaimed = all(ref() is None for ref in cyclic_refs)

    parent, child = Node(), Node()
    parent.peer = child
    child.peer = weakref.ref(parent)
    parent_ref, child_ref = weakref.ref(parent), weakref.ref(child)
    del parent, child
    gc.collect()
    weak_edge_reclaimed = parent_ref() is None and child_ref() is None
    return strong_cycle_reclaimed, weak_edge_reclaimed

In [40]:
assert solve_19() == (True, True)
print("Problem 19: PASS")

Problem 19: PASS


## Problem 20 — Implement an identity-aware map for unhashable objects

**Challenge.** Design `IdentityWeakMap` for **weak-referenceable but unhashable** instance keys, with `set`, `get`, `delete`, and `__len__`. Use integer `id` keys with `weakref.ref` values, verify referent identity on lookup, and protect against an old weakref callback deleting a newer entry if an integer id is reused. Values must not point strongly back to their owners.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 20 — reasoning

A `WeakKeyDictionary` is unsuitable for unhashable objects, but an ordinary `weakref.ref` can still track many unhashable instances. Store `(weakref, value)` under `id(owner)`; compare `ref() is owner` on lookup and `current_ref is dead_ref` in the callback to defend against id reuse. This is a specialized non-thread-safe teaching implementation; concurrent production access needs locks.

In [41]:
class IdentityWeakMap:
    """Identity-based weak-key storage for weak-referenceable objects.

    Not thread-safe. Values must not strongly refer back to their keys.
    """

    def __init__(self):
        self._entries: dict[int, tuple[weakref.ReferenceType, Any]] = {}

    def set(self, owner, value):
        identifier = id(owner)
        entries = self._entries

        def discard(dead_ref):
            current = entries.get(identifier)
            if current is not None and current[0] is dead_ref:
                entries.pop(identifier, None)

        entries[identifier] = (weakref.ref(owner, discard), value)

    def get(self, owner):
        record = self._entries.get(id(owner))
        if record is None or record[0]() is not owner:
            raise KeyError(owner)
        return record[1]

    def delete(self, owner):
        self.get(owner)                # validate identity before removal
        del self._entries[id(owner)]

    def __len__(self):
        return len(self._entries)


def solve_20():
    @dataclass
    class Unhashable:
        name: str

    storage = IdentityWeakMap()
    a, b = Unhashable('same'), Unhashable('same')
    assert a == b and a is not b
    storage.set(a, 10)
    storage.set(b, 20)
    independent = (len(storage), storage.get(a), storage.get(b))
    a_ref, b_ref = weakref.ref(a), weakref.ref(b)
    del a
    gc.collect()
    after_a = (len(storage), a_ref() is None, storage.get(b))
    storage.delete(b)
    del b
    gc.collect()
    return independent, after_a, len(storage), b_ref() is None

In [42]:
independent, after_a, end_size, b_dead = solve_20()
assert independent == (2, 10, 20)
assert after_a == (1, True, 20)
assert end_size == 0 and b_dead
print("Problem 20: PASS")

Problem 20: PASS


## Problem 21 — Descriptor for unhashable, slotted owners

**Challenge.** Build a positive-value descriptor that stores state in the `IdentityWeakMap` from Problem 20. Apply it to a slotted owner with `__weakref__` but no `__dict__`, explicitly make that owner unhashable with `__hash__ = None`, and verify automatic cleanup.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 21 — reasoning

This is the intersection of three constraints: weakref support, unhashability, and no instance dictionary. A custom identity map is one possible solution. The descriptor performs validation and delegates lifetime-safe storage to the preceding problem. This solution intentionally depends on `IdentityWeakMap` from Problem 20.

In [43]:
class WeakIdentityField:
    def __init__(self):
        self._store = IdentityWeakMap()
        self._name = '<unset>'

    def __set_name__(self, owner, name):
        self._name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._store.get(instance)
        except KeyError:
            raise AttributeError(f'{self._name} is not set') from None

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError('expected a non-boolean integer')
        if value <= 0:
            raise ValueError('expected a positive integer')
        self._store.set(instance, value)

    def __delete__(self, instance):
        try:
            self._store.delete(instance)
        except KeyError:
            raise AttributeError(f'{self._name} is not set') from None


class SlottedAccount:
    __slots__ = ('name', '__weakref__')
    __hash__ = None
    balance = WeakIdentityField()

    def __init__(self, name):
        self.name = name


def solve_21():
    owner = SlottedAccount('A')
    owner.balance = 42
    before = (owner.balance, not hasattr(owner, '__dict__'))
    try:
        hash(owner)
    except TypeError:
        unhashable = True
    else:
        unhashable = False
    ptr = weakref.ref(owner)
    del owner
    gc.collect()
    return before, unhashable, ptr() is None, len(SlottedAccount.balance._store)

In [44]:
assert solve_21() == ((42, True), True, True, 0)
print("Problem 21: PASS")

Problem 21: PASS


## Problem 22 — Choose `__dict__` instead of weak maps when possible

**Challenge.** Implement a descriptor using unique names in `instance.__dict__` rather than a class-level mapping. Show that default dataclass instances may be unhashable but still work. Compare this design’s requirements with Problems 8 and 21.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 22 — reasoning

Per-instance dictionaries are often the simplest descriptor storage: they naturally follow each instance’s lifetime, require no weakref or hashability, and do not put a central map on the descriptor. They are not available for instances with restrictive slots and no dictionary.

In [45]:
class LocalField:
    def __set_name__(self, owner, name):
        self.storage_name = f'__managed_{name}'

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(self.storage_name) from None

    def __set__(self, instance, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError('nonempty string required')
        instance.__dict__[self.storage_name] = value

    def __delete__(self, instance):
        try:
            del instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(self.storage_name) from None


@dataclass
class Profile:
    username: str
    nickname = LocalField()


def solve_22():
    person = Profile('pythonista')
    person.nickname = 'Py'
    first = person.nickname
    del person.nickname
    try:
        _ = person.nickname
    except AttributeError:
        missing = True
    else:
        missing = False
    try:
        hash(person)
    except TypeError:
        unhashable = True
    else:
        unhashable = False
    return first, missing, unhashable

In [46]:
assert solve_22() == ('Py', True, True)
print("Problem 22: PASS")

Problem 22: PASS


## Problem 23 — A weak-key memory leak through the value

**Challenge.** Demonstrate a problematic `WeakKeyDictionary` entry whose value is a dictionary containing a **strong reference to its own key**. Show that deleting the external owner does not evict the entry until the mapping is cleared. Then implement the safe alternative storing a weak back-reference in the value.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 23 — reasoning

Weak keys do not save you when a value reachable from the mapping points strongly to the key. The strong chain `mapping -> value -> owner` defeats weak-key eviction. Replace such back-references with `weakref.ref(owner)` or redesign the data ownership graph.

In [47]:
def solve_23():
    class Owner:
        pass

    broken = weakref.WeakKeyDictionary()
    owner = Owner()
    witness = weakref.ref(owner)
    broken[owner] = {'owner': owner, 'payload': 1}
    del owner
    gc.collect()
    kept_alive = (witness() is not None, len(broken))
    broken.clear()
    gc.collect()
    cleared = witness() is None

    safe = weakref.WeakKeyDictionary()
    second = Owner()
    second_ref = weakref.ref(second)
    safe[second] = {'owner_ref': weakref.ref(second), 'payload': 2}
    del second
    gc.collect()
    safe_evicted = (second_ref() is None, len(safe))
    return kept_alive, cleared, safe_evicted

In [48]:
assert solve_23() == ((True, 1), True, (True, 0))
print("Problem 23: PASS")

Problem 23: PASS


## Problem 24 — Hashing a weak reference after death

**Challenge.** Investigate two weak refs to distinct hashable objects: hash the first while alive but do **not** hash the second until it is dead. What happens when you hash them after collection? Why is a dead weakref not a safe universal replacement for key identity?

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 24 — reasoning

`hash(weakref)` normally reflects the referent’s hash and caches it on first use. If the first hash happens only **after** death, `TypeError` is raised. Weakref equality/hash also follow object semantics rather than guaranteeing unique identity mapping; see Problems 9 and 20.

In [49]:
def solve_24():
    class Key:
        pass

    first, second = Key(), Key()
    ref_hashed = weakref.ref(first)
    ref_unhashed = weakref.ref(second)
    prior_hash = hash(ref_hashed)
    del first, second
    gc.collect()
    repeat_hash = hash(ref_hashed)
    try:
        hash(ref_unhashed)
    except TypeError:
        unhashed_dead_raises = True
    else:
        unhashed_dead_raises = False
    return prior_hash == repeat_hash, unhashed_dead_raises

In [50]:
assert solve_24() == (True, True)
print("Problem 24: PASS")

Problem 24: PASS


## Problem 25 — Deterministic two-thread lifetime handoff

**Challenge.** Coordinate two threads with `threading.Event`. A worker obtains a referent from `ref()` and keeps it in a local while the main thread deletes its strong owner. Verify the object stays alive until the worker releases its local. Avoid sleep-based race tests.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 25 — reasoning

The result of `ref()` temporarily becomes a strong owner. A single successful dereference is sufficient to protect the object **during use**. Events make the handoff order explicit; no time-of-check/time-of-use race or arbitrary `sleep()` is needed.

In [51]:
def solve_25():
    class Payload:
        pass

    owner = Payload()
    watcher = weakref.ref(owner)
    acquired = threading.Event()
    release = threading.Event()
    results = []

    def worker():
        local = watcher()             # worker owns the referent while it processes it
        results.append(local is not None)
        acquired.set()
        assert release.wait(timeout=10), 'worker release timed out'
        del local

    thread = threading.Thread(target=worker)
    thread.start()
    try:
        assert acquired.wait(timeout=10), 'worker acquisition timed out'
        del owner
        gc.collect()
        still_alive = watcher() is not None
    finally:
        release.set()
        thread.join(timeout=10)
    assert not thread.is_alive(), 'worker did not finish'
    gc.collect()
    return results, still_alive, watcher() is None

In [52]:
assert solve_25() == ([True], True, True)
print("Problem 25: PASS")

Problem 25: PASS


## Problem 26 — Capstone: two-tier cache, explicit close, and slots

**Challenge.** Implement a weak-value cache with an LRU-sized *strong* window (limit 2), so recently used resources stay cached while older resources may survive if external clients own them. Also implement a `ManagedResource` using `weakref.finalize` for idempotent explicit cleanup, and optionally demonstrate dataclass `weakref_slot=True` on Python 3.11+.

**Work first:** try writing your own solution before expanding/running the solution cell. Every problem has executable assertions, and examples are intentionally scoped to functions to avoid stray notebook references.

### Solution 26 — reasoning

Weak-only caches have no retention guarantee; mixing a bounded strong window and weak values gives controlled ownership. For cleanup, register `weakref.finalize(self, independent_handle.close)` where the handle does not point back to `self`, and call the finalizer in `close()` to run it at most once. Modern slotted dataclasses require `weakref_slot=True` (Python 3.11+) when weak references are desired.

In [53]:
class Resource:
    def __init__(self, key):
        self.key = key


class TwoTierCache:
    def __init__(self, strong_limit=2):
        if strong_limit < 1:
            raise ValueError('strong_limit must be positive')
        self.weak = weakref.WeakValueDictionary()
        self.strong = OrderedDict()
        self.limit = strong_limit
        self.created = 0

    def get(self, key):
        result = self.strong.pop(key, None)
        if result is None:
            result = self.weak.get(key)
        if result is None:
            result = Resource(key)
            self.weak[key] = result
            self.created += 1
        self.strong[key] = result
        if len(self.strong) > self.limit:
            self.strong.popitem(last=False)  # evict oldest STRONG ownership
        return result


class IndependentHandle:
    def __init__(self, events):
        self.events = events

    def close(self):
        self.events.append('closed')


class ManagedResource:
    def __init__(self, events):
        handle = IndependentHandle(events)       # does not retain ManagedResource
        self._finalizer = weakref.finalize(self, IndependentHandle.close, handle)

    def close(self):
        self._finalizer()                        # idempotent


def solve_26():
    cache = TwoTierCache(strong_limit=2)
    a = cache.get('a')
    cache.get('b')
    cache.get('c')                             # 'a' evicted from strong window
    still_weakly_cached = cache.weak.get('a') is a
    reused = cache.get('a') is a              # 'a' gets promoted
    assert list(cache.strong) == ['c', 'a']
    assert len(cache.strong) == 2
    del a

    events = []
    owned = ManagedResource(events)
    observer = weakref.ref(owned)
    owned.close()
    owned.close()
    del owned
    gc.collect()

    # The following functionality was added to dataclasses in Python 3.11.
    optional_slots = 'skipped on Python < 3.11'
    if sys.version_info >= (3, 11):
        @dataclass(slots=True, weakref_slot=True)
        class SlottedDataclass:
            value: int

        instance = SlottedDataclass(9)
        optional_slots = weakref.ref(instance)() is instance

    return (still_weakly_cached, reused, cache.created,
            events, observer() is None, optional_slots)

In [54]:
result = solve_26()
assert result[:5] == (True, True, 3, ['closed'], True)
assert result[5] is True if sys.version_info >= (3, 11) else result[5] == 'skipped on Python < 3.11' 
print("Problem 26: PASS")

Problem 26: PASS


## Capstone comparison table — pick a storage mechanism deliberately

| Mechanism | Keys owned strongly? | Values owned strongly? | Key constraints | Typical use / caveat |
|---|---|---|---|---|
| Normal `dict` | Yes | Yes | Hashable | Explicit ownership; a descriptor-level map may retain instances forever. |
| `weakref.ref(obj)` | No referent ownership | N/A | Weak-referenceable object | One-off observation; use one local after calling `ref()`. |
| `WeakKeyDictionary` | No | Yes | Weak-referenceable **and hashable** | Per-instance metadata when key equality/hashing fits; values must not own keys. |
| `WeakValueDictionary` | Yes (normal keys) | No | Weak-referenceable **values** | Reusable cache *only while somebody else owns values*. |
| `WeakSet` | No members owned strongly | N/A | Weak-referenceable and hashable | Non-owning observer/subscriber sets. |
| `WeakMethod` | Does not own bound instance | N/A | Supported bound methods | Event handler references that respect instance lifetime. |
| `instance.__dict__` | Stored on instance | Yes, by instance | Requires instance dictionary | Straightforward descriptor storage; no hashability needed. |
| Custom `IdentityWeakMap` | No (tracked by weakref) | Yes | Weak-referenceable, **unhashable allowed** | Specialized identity store; guard `id()` reuse and synchronize threads. |

**Checklist before shipping:** Who owns the resource? Can values refer back to weak keys? Are equals-but-distinct instances supposed to share state? Is hashing stable? Is the class slotted? Are callbacks/finalizers owner-free? Does concurrency require a lock? Do tests inadvertently retain the object?

## Extra advanced challenges (try before the answer key)

1. **Broken observer design:** Why does putting subscribers in a plain set change object lifetime? Fix with `WeakSet` and explain when the *caller* must retain a subscriber.
2. **Dead references:** Is `weakref.ref(x)() is None` always a perfect existence test? Explain how a valid referent whose own value is `None` differs from a weak reference to an object. (Hint: Python `None` itself is not a typical weak-referenceable referent.)
3. **Descriptor equality:** Two logically equal user records need *different* validated values. Which is safer: value-based hashing with `WeakKeyDictionary`, `eq=False` identity-based records, `instance.__dict__`, or a custom identity weak map? State assumptions.
4. **Cache miss after write:** Why can `cache['x'] = Resource('x')` followed by `cache['x']` raise `KeyError` on CPython when `cache` is a `WeakValueDictionary`?
5. **Finalizer cycles:** Explain what changes if the cleanup callback is a bound method of the owner rather than a method of an independent handle.
6. **Lifecycle tests:** Why can the final expression of a Jupyter cell make a weakref-death test fail? Show a strategy that returns only booleans, strings, and integers.
7. **Slots:** Why are `__slots__ = ('state',)` and default `@dataclass(slots=True)` insufficient for weak references in many cases? What options enable them?
8. **Concurrency:** Show how a single local `value = ref()` prevents the referent from dying midway through `value.work()` on an interpreter with normal strong ownership semantics.

### Extra challenge answer key

1. A normal set owns subscribers strongly; `WeakSet` does not, so the caller must keep desired listeners alive.
2. `ref()` is the safe existence probe; use `is None` rather than truthiness because live referents might be falsey. `None` is a death sentinel rather than an arbitrary payload here.
3. Use `instance.__dict__` if available; use `eq=False` identity keys with `WeakKeyDictionary` if hashability/weakref support are appropriate; use `IdentityWeakMap` if unhashable slotted objects need central storage. Avoid equal-key collisions.
4. The temporary resource has no external strong owner; the cache retains only a weak value and it may be reclaimed before the next lookup.
5. A bound callback can keep the owner strongly alive and prevent its own finalization. Pass an independent handle instead.
6. IPython output history may keep the displayed referent. Run ownership tests in functions and return only primitive summaries; do not expose objects as final expression results.
7. Add `__weakref__` to manual slots, or use `@dataclass(slots=True, weakref_slot=True)` on Python 3.11+; a suitable base class can also provide weakref support.
8. Copy `ref()` to a local once; while the local lives it is a strong reference, so a concurrent drop of other owners cannot destroy the object before the operation finishes. This does *not* make the object’s mutable state thread-safe.

## Final self-check / source boundaries

The reference material supports the foundational behavior of weak references and weak-key descriptors, including hashability limitations. Problems on weak values, weak sets, weak bound methods, proxies, finalizers, cycles, cache architectures, slots, hash mutation, and threads are **additional applied examples** built to deepen that topic; they are not claimed to appear in the supplied lesson. Assertions intentionally verify behavior without printing live referents or reading freed memory addresses.

**Suggested practice:** restart the kernel, solve odd-numbered challenges from scratch, run the supplied tests, then solve the even-numbered challenges without looking at their solutions.